# MK8: HMC for CW Parameters in Hybrid Sampler

## Problem
Same as MK5: 13 parameters (8 CW + 5 distances), phase-connected model creates
hundreds of modes per prior sigma in distance space.

## What's new: HMC replaces MH eigenmode proposals for CW params

**MK5 bottleneck**: CW ESS stuck at ~20 because MH eigenmode proposals can't efficiently
explore curved correlations (modes 3, 6, 7 had low AR even with adaptive scaling).

**MK8 approach**: Use Hamiltonian Monte Carlo (HMC) for the 8 CW parameters,
keeping all distance moves unchanged:

| Move | Probability | Description |
|------|------------|-------------|
| **HMC CW** | 40% | Leapfrog integration on 8 CW dims with Fisher mass matrix |
| **Distance within-mode** | 20% | Small 1D refinement using Fisher width |
| **Prior-snap big jump** | 30% | Draw from EM prior, snap to nearest mode peak |
| **Coherent freq-dist** | 10% | Joint frequency-distance proposal with Newton snapping |

**HMC details**:
- Mass matrix M = -H_cw (Fisher precision) — matches posterior curvature
- Leapfrog integrator with L steps (configurable, default 10)
- Step size tuned via dual averaging during burn-in (target AR ~0.70)
- Gradients from JAX autodiff (already available)

In [1]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"
os.environ["JAX_DISABLE_MMAP_CACHE"] = "1"
os.environ["XLA_FLAGS"] = "--xla_gpu_autotune_level=2"

import glob, time
import numpy as np
import matplotlib.pyplot as plt

import jax
jax.config.update('jax_enable_x64', True)
jax.clear_caches()
import jax.numpy as jnp

import discovery as ds
from enterprise_extensions import load_feathers
from discovery.deterministic import make_phase_connected_binary
from discovery import const as disco_const
from discovery.deterministic import fpcmu_fast

print('Imports OK')

/home/mattm/miniforge3/envs/discotech/lib/python3.12/site-packages/enterprise/signals/utils.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import Requirement, resource_filename


Imports OK


In [2]:
# Load pulsars and their EM distance priors
feather_dir = "../data_products/"
Npulsars = 5

disco_psrs = [ds.Pulsar.read_feather(f) for f in sorted(glob.glob(feather_dir + "*.feather"))][:Npulsars]
for psr in disco_psrs:
    psr.toaerrs = np.full_like(psr.toas, 1e-6, dtype=np.float32)
print(f"Loaded {len(disco_psrs)} pulsars: {[p.name for p in disco_psrs]}")

psrs_ent = load_feathers.load_feathers_from_folder(feather_dir)
ent_by_name = {p.name: p for p in psrs_ent}

dist_mu = []
dist_sig = []
for psr in disco_psrs:
    ep = ent_by_name[psr.name]
    mu = float(ep.pdist[0])
    sig = float(ep.pdist[1]) if len(ep.pdist) > 1 else 0.5
    if (not np.isfinite(sig)) or sig <= 0:
        sig = 0.5
    dist_mu.append(mu)
    dist_sig.append(sig)

dist_mu = jnp.array(dist_mu, dtype=jnp.float64)
dist_sig = jnp.array(dist_sig, dtype=jnp.float64)

psr_toas_list = [np.asarray(psr.toas, dtype=np.float64) for psr in disco_psrs]
psr_pos_list = [psr.pos for psr in disco_psrs]
psr_positions = jnp.array([psr.pos for psr in disco_psrs])

sigma_toa = 1e-6
KPC_OVER_C = disco_const.kpc / disco_const.c

pnames_13 = (['cos_gwtheta', 'gwphi', 'cos_inc', 'log10_mc', 'log10_fgw',
              'log10_h', 'phase0', 'psi'] +
             [psr.name + '_dist' for psr in disco_psrs])

print(f"dist_mu: {[f'{float(d):.3f}' for d in dist_mu]}")
print(f"dist_sig: {[f'{float(d):.4f}' for d in dist_sig]}")

Loaded 5 pulsars: ['B1855+09', 'B1937+21', 'B1953+29', 'J0023+0923', 'J0030+0451']
FeatherPulsar.read_feather: cannot find dmx in feather file ../data_products/B1855+09.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file ../data_products/B1855+09.feather.
FeatherPulsar.read_feather: cannot find dmx in feather file ../data_products/B1937+21.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file ../data_products/B1937+21.feather.
FeatherPulsar.read_feather: cannot find dmx in feather file ../data_products/B1953+29.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file ../data_products/B1953+29.feather.
FeatherPulsar.read_feather: cannot find dmx in feather file ../data_products/J0023+0923.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file ../data_products/J0023+0923.feather.
FeatherPulsar.read_feather: cannot find dmx in feather file ../data_products/J0030+0451.feather.
FeatherPulsar.read_feather: cannot find _pdi

In [3]:
# CW injection and log-posterior definition
INJ = {
    "cos_gwtheta": 0.3, "gwphi": 2.5, "cos_inc": -0.2,
    "log10_mc": 9.0, "log10_fgw": -8.0, "log10_h": -12.0,
    "phase0": 1.0, "psi": 0.7,
}

cw_func = make_phase_connected_binary(pulsarterm=True)

@jax.jit
def compute_delta_L(cos_gwtheta, gwphi, log10_fgw):
    """Mode spacing in distance (kpc) for each pulsar."""
    gwtheta = jnp.arccos(cos_gwtheta)
    f_gw = 10.0 ** log10_fgw
    _, _, cos_mu = jax.vmap(
        lambda pos: fpcmu_fast(pos, gwtheta, gwphi)
    )(psr_positions)
    denom = jnp.abs(1.0 - cos_mu)
    denom = jnp.maximum(denom, 1e-4)
    return 1.0 / (f_gw * KPC_OVER_C * denom)

# Inject with distances offset from prior mean by 0.3 sigma
DIST_OFFSET_SIGMA = 0.3
p5_true_dists = {}
for i, psr in enumerate(disco_psrs):
    p5_true_dists[psr.name] = float(dist_mu[i]) + DIST_OFFSET_SIGMA * float(dist_sig[i])

p5_data_list = []
for i, psr in enumerate(disco_psrs):
    toas_i = np.asarray(psr.toas, dtype=np.float64)
    delay_i = cw_func(toas_i, psr.pos, p_dist=p5_true_dists[psr.name], **INJ)
    p5_data_list.append(np.array(delay_i, dtype=np.float64))

INJ_vals = [INJ[k] for k in ['cos_gwtheta','gwphi','cos_inc','log10_mc','log10_fgw','log10_h','phase0','psi']]
p5_dist_vals = [p5_true_dists[psr.name] for psr in disco_psrs]
p5_truth = np.array(INJ_vals + p5_dist_vals)

sd_arr = np.array([float(dist_sig[i]) for i in range(Npulsars)])
mu_arr = np.array([float(dist_mu[i]) for i in range(Npulsars)])

# Log-posterior
sd_jnp = jnp.array(sd_arr, dtype=jnp.float64)
data_jnp = [jnp.array(d) for d in p5_data_list]

@jax.jit
def logp(x):
    p_dists = x[8:13]
    in_bounds = (
        (x[0] >= -1) & (x[0] <= 1) &
        (x[1] >= 0) & (x[1] <= 2*jnp.pi) &
        (x[2] >= -1) & (x[2] <= 1) &
        (x[3] >= 7) & (x[3] <= 10) &
        (x[4] >= -9) & (x[4] <= -7) &
        (x[5] >= -18) & (x[5] <= -11) &
        (x[6] >= 0) & (x[6] <= 2*jnp.pi) &
        (x[7] >= 0) & (x[7] <= jnp.pi) &
        jnp.all(p_dists > 1e-6)
    )
    ll = 0.0
    for p_idx in range(5):
        model = cw_func(
            psr_toas_list[p_idx], psr_pos_list[p_idx],
            cos_gwtheta=x[0], gwphi=x[1], cos_inc=x[2],
            log10_mc=x[3], log10_fgw=x[4], log10_h=x[5],
            phase0=x[6], psi=x[7], p_dist=p_dists[p_idx], p_phase=None
        )
        resid = data_jnp[p_idx] - model
        ll -= 0.5 * jnp.sum(resid**2) / sigma_toa**2
    log_prior = -0.5 * jnp.sum(jnp.square((p_dists - dist_mu) / sd_jnp))
    return jnp.where(in_bounds, ll + log_prior, -1e30)

# Mode spacing at truth
dL_truth = np.array(compute_delta_L(INJ['cos_gwtheta'], INJ['gwphi'], INJ['log10_fgw']))

lp_truth = float(logp(jnp.array(p5_truth)))
print(f"logp(truth) = {lp_truth:.2f}")
for i, psr in enumerate(disco_psrs):
    print(f"  {psr.name}: d_true={p5_dist_vals[i]:.4f}, dL={dL_truth[i]:.6f}, "
          f"modes/sig={float(dist_sig[i])/dL_truth[i]:.0f}")

logp(truth) = -0.23
  B1855+09: d_true=1.2160, dL=0.000580, modes/sig=207
  B1937+21: d_true=3.1600, dL=0.000583, modes/sig=343
  B1953+29: d_true=4.9190, dL=0.000603, modes/sig=1542
  J0023+0923: d_true=1.0530, dL=0.000591, modes/sig=186
  J0030+0451: d_true=0.3307, dL=0.000587, modes/sig=6


In [ ]:
# Compute Fisher information at any on-peak point.
#
# The Hessian is MODE-INVARIANT (verified in MK4: CW block mean rel diff
# = 2.6%, distance diagonals < 4%). So we compute it at whatever starting
# point we have — doesn't need to be the truth.

# JIT-compiled gradient for Newton snapping and HMC leapfrog
grad_logp = jax.jit(jax.grad(logp))

def compute_fisher(x_peak, logp_fn):
    """Compute CW proposal components and distance Fisher widths at a mode peak.
    
    Returns dict with:
      - L_cw: 8x8 Cholesky for joint 8D proposal (kept for reference)
      - eig_vecs: (8,8) eigenvectors of CW covariance (columns)
      - eig_sigs: (8,) per-eigenmode proposal widths (1D optimal scaling)
      - dist_fisher_sig: (Npsr,) per-pulsar within-mode widths
      - cw_gibbs_sig: (8,) per-param Gibbs step sizes
      - H_dist_diag: (Npsr,) Hessian diagonal for distance params (for Newton snap)
      - cov_cw: (8,8) CW covariance matrix (M^{-1} for HMC)
      - M_chol: (8,8) Cholesky of mass matrix M = -H_cw (for momentum sampling in HMC)
    """
    print("  Computing Hessian...")
    t0 = time.time()
    H_full = np.array(jax.hessian(logp_fn)(jnp.array(x_peak)))
    dt = time.time() - t0
    print(f"  Hessian computed in {dt:.1f}s")
    
    Npsr = len(x_peak) - 8
    
    # 8x8 CW sub-block
    H_cw = H_full[:8, :8]
    neg_H_cw = -H_cw
    eig_cw, evec_cw = np.linalg.eigh(neg_H_cw)
    eig_cw_c = np.maximum(eig_cw, 1e-12 * eig_cw.max())
    cov_cw = evec_cw @ np.diag(1.0 / eig_cw_c) @ evec_cw.T
    cov_cw = 0.5 * (cov_cw + cov_cw.T)
    
    # Mass matrix M = -H_cw (Fisher precision), regularized
    M = evec_cw @ np.diag(eig_cw_c) @ evec_cw.T
    M = 0.5 * (M + M.T)
    M_chol = np.linalg.cholesky(M)
    
    # Joint 8D Cholesky (kept for reference/fallback)
    scale_8d = 2.38**2 / 8
    L_cw = np.linalg.cholesky(scale_8d * cov_cw)
    
    # Eigenmode proposals (kept for reference)
    eig_vals_cov, eig_vecs_cov = np.linalg.eigh(cov_cw)
    scale_1d = 2.38
    eig_sigs = scale_1d * np.sqrt(np.maximum(eig_vals_cov, 1e-30))
    
    print(f"\n  CW Fisher eigenvalues (precision): {eig_cw}")
    print(f"  Eigenmode proposal widths: {eig_sigs}")
    print(f"  Eigenvalue range: {eig_cw.max()/eig_cw.min():.1e}")
    
    # Per-pulsar distance Fisher widths + Hessian diagonals
    dist_fisher_sig = np.zeros(Npsr)
    H_dist_diag = np.zeros(Npsr)
    dL_at_peak = np.array(compute_delta_L(x_peak[0], x_peak[1], x_peak[4]))
    for j in range(Npsr):
        H_dist_diag[j] = H_full[8+j, 8+j]  # negative (concave)
        neg_hess_jj = -H_dist_diag[j]
        if neg_hess_jj > 0:
            dist_fisher_sig[j] = 1.0 / np.sqrt(neg_hess_jj)
        else:
            dist_fisher_sig[j] = dL_at_peak[j] * 0.3
    
    cw_gibbs_sig = np.diag(L_cw).copy()
    
    for j in range(Npsr):
        print(f"  Pulsar {j}: Fisher sig_d={dist_fisher_sig[j]:.6f}, "
              f"dL={dL_at_peak[j]:.6f}, ratio={dist_fisher_sig[j]/dL_at_peak[j]:.3f}")
    
    return dict(L_cw=L_cw, eig_vecs=eig_vecs_cov, eig_sigs=eig_sigs,
                dist_fisher_sig=dist_fisher_sig, cw_gibbs_sig=cw_gibbs_sig,
                H_dist_diag=H_dist_diag, cov_cw=cov_cw, M_chol=M_chol)

# Compute at truth for Test 1
print("Fisher at truth:")
fisher = compute_fisher(p5_truth, logp)
L_cw = fisher['L_cw']
dist_fisher_sig = fisher['dist_fisher_sig']
cw_gibbs_sig = fisher['cw_gibbs_sig']
eig_vecs = fisher['eig_vecs']
eig_sigs = fisher['eig_sigs']
print("Done")

In [ ]:
# ============================================================
# Blocked MCMC sampler with:
#   - HMC for 8 CW parameters (replaces eigenmode + joint MH)
#   - Prior-snap big jumps for distances
#   - Coherent frequency-distance proposal using Newton snap
#   - Within-mode distance refinement
# ============================================================

def snap_to_peak(x13, pulsar_idx, dL_j, n_pts=50):
    """Find the nearest mode peak for one pulsar via fine 1D scan."""
    d_center = x13[8 + pulsar_idx]
    d_lo = max(d_center - 0.6 * dL_j, 1e-6)
    d_hi = d_center + 0.6 * dL_j
    d_candidates = np.linspace(d_lo, d_hi, n_pts)
    x_batch = np.tile(x13, (len(d_candidates), 1))
    x_batch[:, 8 + pulsar_idx] = d_candidates
    lps_scan = np.array(jax.vmap(logp)(jnp.array(x_batch)))
    return float(d_candidates[np.argmax(lps_scan)])


def ess_1d(x, max_lag=500):
    """Effective sample size from autocorrelation."""
    n = len(x)
    x = x - x.mean()
    var = np.var(x)
    if var < 1e-30:
        return 1.0
    acf = np.correlate(x, x, mode='full')[n-1:n-1+max_lag+1] / (n * var)
    cutoff = np.argmax(acf < 0.05)
    if cutoff == 0:
        cutoff = max_lag
    tau = 1.0 + 2.0 * np.sum(acf[1:cutoff])
    return n / max(tau, 1.0)


# CW parameter bounds for HMC boundary checking
CW_BOUNDS_LO = np.array([-1.0, 0.0, -1.0, 7.0, -9.0, -18.0, 0.0, 0.0])
CW_BOUNDS_HI = np.array([1.0, 2*np.pi, 1.0, 10.0, -7.0, -11.0, 2*np.pi, np.pi])


def run_sampler(logp_fn, x0, rng, fisher_dict, dL_arr, mu_arr, sig_arr,
                n_burn=3000, n_prod=10000, report_every=2000,
                # Move probabilities (HMC replaces cw_eigen + cw_joint)
                p_hmc=0.40, p_dist_within=0.20,
                p_big_jump=0.30, p_freq_dist=0.10,
                # HMC parameters
                hmc_L=20, hmc_eps_init=0.1, hmc_jitter=True,
                # Dual averaging for HMC step size (burn-in only)
                hmc_target_ar=0.65, hmc_da_gamma=0.05, hmc_da_t0=10, hmc_da_kappa=0.75,
                # Coherent freq-dist proposal width (tuned for ~44% AR)
                freq_dist_sigma=3e-4,
                # Number of Newton iterations for distance correction
                n_newton=3,
                # Hessian recomputation: at what fraction of burn-in
                recompute_hessian_at=0.4):
    """Blocked sampler with HMC for CW params, prior-snap big jumps,
    and coherent frequency-distance proposals with Newton snapping.
    
    HMC move:
      Leapfrog integration on the 8 CW dimensions, holding distances fixed.
      Mass matrix M = -H_cw (Fisher precision from Hessian).
      Step size tuned via dual averaging during burn-in.
      Trajectory length jittered: L ~ Uniform(L//2, 3L//2) to prevent periodic orbits.
    """
    D = len(x0)
    Npsr = D - 8
    x = x0.copy().astype(np.float64)
    lp = float(logp_fn(jnp.array(x)))
    
    # Unpack Fisher components
    dist_fisher_sig = fisher_dict['dist_fisher_sig'].copy()
    H_dist_diag = fisher_dict['H_dist_diag'].copy()
    cov_cw = fisher_dict['cov_cw'].copy()   # M^{-1} for HMC leapfrog
    M_chol = fisher_dict['M_chol'].copy()    # Cholesky of M for momentum sampling
    
    chain = np.zeros((n_prod, D))
    lps = np.zeros(n_prod)
    
    acc = {'hmc': 0, 'dist_within': 0, 'big_jump': 0, 'freq_dist': 0}
    tot = {k: 0 for k in acc}
    
    # HMC step size and dual averaging state
    log_eps = np.log(hmc_eps_init)
    log_eps_bar = 0.0  # running average
    H_bar = 0.0        # running average of acceptance stat
    mu_da = np.log(10 * hmc_eps_init)  # bias term
    hmc_step_count = 0
    
    # Track gradient evals for cost comparison
    n_grad_evals = 0
    n_logp_evals = 0
    
    t1 = p_hmc
    t2 = t1 + p_dist_within
    t3 = t2 + p_big_jump
    
    total_steps = n_burn + n_prod
    recompute_step = int(recompute_hessian_at * n_burn)
    hessian_recomputed = False
    t0 = time.time()
    
    for step in range(-n_burn, n_prod):
        abs_step = step + n_burn
        
        # --- Hessian recomputation at mid-burn-in ---
        if not hessian_recomputed and abs_step == recompute_step:
            print(f"  [Recomputing Hessian at burn-in step {abs_step}]")
            x_snap = x.copy()
            for j in range(Npsr):
                dL_j = float(np.array(compute_delta_L(x[0], x[1], x[4]))[j])
                x_snap[8+j] = snap_to_peak(x_snap, j, dL_j)
            
            new_fisher = compute_fisher(x_snap, logp_fn)
            dist_fisher_sig = new_fisher['dist_fisher_sig']
            H_dist_diag = new_fisher['H_dist_diag']
            cov_cw = new_fisher['cov_cw']
            M_chol = new_fisher['M_chol']
            hessian_recomputed = True
            print(f"  [Hessian recomputed, HMC mass matrix updated]")
        
        r = rng.random()
        
        if r < t1:
            # --- HMC on 8 CW dimensions ---
            eps = np.exp(log_eps)
            q = x[:8].copy()       # CW params (position)
            p = M_chol @ rng.standard_normal(8)  # momentum ~ N(0, M)
            
            # Jitter trajectory length to prevent periodic orbits
            if hmc_jitter:
                L_this = rng.integers(max(1, hmc_L // 2), max(2, 3 * hmc_L // 2) + 1)
            else:
                L_this = hmc_L
            
            # Current Hamiltonian: H = -logp + 0.5 * p^T M^{-1} p
            kinetic_old = 0.5 * p @ (cov_cw @ p)
            H_old = -lp + kinetic_old
            
            # Leapfrog integration
            x_full = x.copy()
            q_new = q.copy()
            p_new = p.copy()
            
            # Half step for momentum
            x_full[:8] = q_new
            g = np.array(grad_logp(jnp.array(x_full)))[:8]
            n_grad_evals += 1
            p_new += 0.5 * eps * g
            
            # Full steps
            divergent = False
            for l in range(L_this):
                q_new += eps * (cov_cw @ p_new)
                
                # Check bounds — if out of bounds, reject trajectory
                if np.any(q_new < CW_BOUNDS_LO) or np.any(q_new > CW_BOUNDS_HI):
                    divergent = True
                    break
                
                x_full[:8] = q_new
                g = np.array(grad_logp(jnp.array(x_full)))[:8]
                n_grad_evals += 1
                
                if l < L_this - 1:
                    p_new += eps * g
                else:
                    p_new += 0.5 * eps * g  # half step at end
            
            if not divergent:
                # Proposed Hamiltonian
                x_prop = x.copy()
                x_prop[:8] = q_new
                lp_prop = float(logp_fn(jnp.array(x_prop)))
                n_logp_evals += 1
                kinetic_new = 0.5 * p_new @ (cov_cw @ p_new)
                H_new = -lp_prop + kinetic_new
                
                delta_H = H_new - H_old
                alpha = min(1.0, np.exp(-delta_H))
                
                if rng.random() < alpha:
                    x = x_prop; lp = lp_prop
                    acc['hmc'] += 1
            else:
                alpha = 0.0
            
            tot['hmc'] += 1
            
            # Dual averaging for step size (burn-in only)
            if step < 0:
                hmc_step_count += 1
                m = hmc_step_count
                w = 1.0 / (m + hmc_da_t0)
                H_bar = (1.0 - w) * H_bar + w * (hmc_target_ar - alpha)
                log_eps = mu_da - np.sqrt(m) / hmc_da_gamma * H_bar
                log_eps = np.clip(log_eps, -10, 2)  # safety bounds
                m_w = m ** (-hmc_da_kappa)
                log_eps_bar = m_w * log_eps + (1.0 - m_w) * log_eps_bar
            
        elif r < t2:
            # --- Within-mode distance refinement ---
            pi = rng.integers(Npsr)
            x_prop = x.copy()
            x_prop[8+pi] += dist_fisher_sig[pi] * rng.standard_normal()
            n_logp_evals += 1
            if x_prop[8+pi] > 1e-6:
                lp_prop = float(logp_fn(jnp.array(x_prop)))
                if np.log(rng.random() + 1e-300) < lp_prop - lp:
                    x = x_prop; lp = lp_prop; acc['dist_within'] += 1
            tot['dist_within'] += 1
            
        elif r < t3:
            # --- Prior-snap big jump ---
            pi = rng.integers(Npsr)
            d_prop = rng.normal(mu_arr[pi], sig_arr[pi])
            if d_prop > float(dL_arr[pi]):
                x_snap = x.copy()
                x_snap[8+pi] = d_prop
                d_snapped = snap_to_peak(x_snap, pi, float(dL_arr[pi]))
                x_prop = x.copy()
                x_prop[8+pi] = d_snapped
                lp_prop = float(logp_fn(jnp.array(x_prop)))
                n_logp_evals += 1
                if np.log(rng.random() + 1e-300) < lp_prop - lp:
                    x = x_prop; lp = lp_prop; acc['big_jump'] += 1
            tot['big_jump'] += 1
            
        else:
            # --- Coherent frequency-distance proposal (Newton snap) ---
            delta = rng.standard_normal() * freq_dist_sigma
            x_prop = x.copy()
            x_prop[4] += delta
            
            if -9 <= x_prop[4] <= -7:
                scale_factor = 10.0 ** (-delta)
                for j in range(Npsr):
                    x_prop[8 + j] *= scale_factor
                
                for _newton in range(n_newton):
                    g = np.array(grad_logp(jnp.array(x_prop)))
                    n_grad_evals += 1
                    for j in range(Npsr):
                        if H_dist_diag[j] < -1e-6:
                            newton_step = -g[8 + j] / H_dist_diag[j]
                            x_prop[8 + j] += newton_step
                            x_prop[8 + j] = max(x_prop[8 + j], 1e-6)
                
                lp_prop = float(logp_fn(jnp.array(x_prop)))
                n_logp_evals += 1
                if np.log(rng.random() + 1e-300) < lp_prop - lp:
                    x = x_prop; lp = lp_prop; acc['freq_dist'] += 1
            tot['freq_dist'] += 1
        
        if step >= 0:
            chain[step] = x
            lps[step] = lp
        
        # Update mode spacing periodically
        if abs_step > 0 and abs_step % 2000 == 0:
            dL_arr = np.array(compute_delta_L(x[0], x[1], x[4]))
        
        # Progress report
        if abs_step > 0 and abs_step % report_every == 0:
            elapsed = time.time() - t0
            phase = 'burn' if step < 0 else 'prod'
            rate = abs_step / elapsed
            eta = (total_steps - abs_step) / rate
            ar_hmc = acc['hmc'] / max(tot['hmc'], 1)
            ar_bj = acc['big_jump'] / max(tot['big_jump'], 1)
            ar_fd = acc['freq_dist'] / max(tot['freq_dist'], 1)
            cur_eps = np.exp(log_eps)
            print(f"  [{phase} {abs_step}/{total_steps}] lp={lp:.2f} "
                  f"HMC={ar_hmc:.3f}(eps={cur_eps:.4f}) BJ={ar_bj:.3f} FD={ar_fd:.3f} "
                  f"[{rate:.0f} it/s, ETA {eta:.0f}s]")
    
    # Use averaged step size after burn-in
    final_eps = np.exp(log_eps_bar) if hmc_step_count > 0 else np.exp(log_eps)
    
    dt_total = time.time() - t0
    ar_dict = {k: acc[k]/max(tot[k],1) for k in acc}
    print(f"\nDone in {dt_total:.1f}s ({total_steps/dt_total:.0f} it/s)")
    for k in acc:
        print(f"  {k:15s}: {acc[k]:6d}/{tot[k]:6d} = {ar_dict[k]:.4f}")
    print(f"\n  HMC step size: eps={final_eps:.6f} (L={hmc_L}, jitter={hmc_jitter})")
    print(f"  Gradient evals: {n_grad_evals}, logp evals: {n_logp_evals}")
    print(f"  Avg grads/step: {n_grad_evals/total_steps:.1f}")
    
    return chain, lps, ar_dict

print("Sampler defined (HMC CW + jitter + big jumps + freq-dist Newton)")

In [6]:
def run_diagnostics(chain, lps, ar, truth, dL, title='', show_start=None):
    """Print diagnostics and plot traces / posteriors."""
    Npsr = chain.shape[1] - 8
    cw_keys = ['cos_gwtheta', 'gwphi', 'cos_inc', 'log10_mc',
               'log10_fgw', 'log10_h', 'phase0', 'psi']
    
    # Coverage
    cov = sum(1 for k in range(13)
              if np.percentile(chain[:,k], 5) <= truth[k] <= np.percentile(chain[:,k], 95))
    print(f"Coverage: {cov}/13")
    
    # ESS
    ess_vals = {pnames_13[k]: ess_1d(chain[:,k]) for k in range(13)}
    print(f"ESS range: [{min(ess_vals.values()):.0f}, {max(ess_vals.values()):.0f}]")
    for k, v in ess_vals.items():
        print(f"  {k:20s}: ESS={v:.0f}")
    
    # Distance errors
    print(f"\nDistance errors:")
    for j in range(Npsr):
        med = np.median(chain[:, 8+j])
        err_dL = abs(med - truth[8+j]) / dL[j]
        print(f"  {disco_psrs[j].name:13s}: median={med:.6f}, truth={truth[8+j]:.6f}, "
              f"err={err_dL:.1f} dL")
    
    # Acceptance rates
    print(f"\nAcceptance rates:")
    for k, v in ar.items():
        print(f"  {k:15s}: {v:.4f}")
    
    # --- Plots ---
    fig, axes = plt.subplots(3, 3, figsize=(15, 12))
    fig.suptitle(f'{title} (cov={cov}/13)', fontsize=14)
    
    # logp trace
    ax = axes[0, 0]
    ax.plot(lps, color='#1a3a5c', lw=0.4, alpha=0.8)
    ax.axhline(float(logp(jnp.array(truth))), color='r', ls='--', lw=1, label='truth')
    ax.set_xlabel('step'); ax.set_ylabel('logp')
    ax.set_title('Log-posterior trace'); ax.legend(fontsize=8)
    
    # CW param traces
    for i, idx in enumerate([0, 1, 4, 5]):
        ax = axes[0, 1] if i == 0 else axes[0, 2] if i == 1 else axes[1, 0] if i == 2 else axes[1, 1]
        ax.plot(chain[:, idx], color='#1a3a5c', lw=0.4, alpha=0.8)
        ax.axhline(truth[idx], color='r', ls='--', lw=1)
        ax.set_title(cw_keys[idx]); ax.set_xlabel('step')
    
    # Distance traces
    ax = axes[1, 2]
    colours = ['#1a3a5c', '#8b2500', '#2d5a27']
    for j in range(min(3, Npsr)):
        ax.plot(chain[:, 8+j], alpha=0.7, lw=0.4, color=colours[j],
                label=disco_psrs[j].name[:8])
        ax.axhline(truth[8+j], color=colours[j], ls='--', lw=0.8)
        if show_start is not None:
            ax.axhline(show_start[8+j], color=colours[j], ls=':', lw=0.8, alpha=0.5)
    ax.set_xlabel('step'); ax.set_ylabel('dist (kpc)')
    ax.set_title('Distance traces (first 3)'); ax.legend(fontsize=7)
    
    # Distance posteriors vs priors
    for j in range(min(3, Npsr)):
        ax = axes[2, j]
        ax.hist(chain[:, 8+j], bins=80, density=True, alpha=0.7, color='#2b5797')
        ax.axvline(truth[8+j], color='r', ls='--', lw=1.5, label='truth')
        ax.axvline(float(dist_mu[j]), color='green', ls=':', lw=1.5, label='prior mean')
        if show_start is not None:
            ax.axvline(show_start[8+j], color='orange', ls=':', lw=1.5, label='start')
        ax.set_title(f'{disco_psrs[j].name[:8]} dist'); ax.legend(fontsize=7)
    
    plt.tight_layout(); plt.show()
    return cov, ess_vals

print("Diagnostics function defined")

Diagnostics function defined


## Test 1: Start from truth

Baseline test to verify the sampler works when starting at the correct mode.
Expected: high coverage, CW Fisher AR ~0.2, big-jump AR ~0.45.

In [ ]:
print("Test 1: Start from truth (HMC CW proposals)")
print(f"  logp(truth) = {lp_truth:.2f}")

dL_start = np.array(compute_delta_L(p5_truth[0], p5_truth[1], p5_truth[4]))

chain1, lps1, ar1 = run_sampler(
    logp, p5_truth.copy(), np.random.default_rng(42),
    fisher_dict=fisher, dL_arr=dL_start, mu_arr=mu_arr, sig_arr=sd_arr,
    n_burn=3000, n_prod=15000, report_every=2000,
    hmc_L=20, hmc_eps_init=0.1, hmc_target_ar=0.65)

cov1, ess1 = run_diagnostics(chain1, lps1, ar1, p5_truth, dL_start,
                              title='Test 1: Start from truth (HMC L=20)')

## Test 2: Start from a wrong mode (shifted distances)

The real test: can the prior-snap big jumps find the correct mode when
starting hundreds of modes away? We shift each distance by a random number
of modes (staying within 2-sigma of the EM prior), then snap to the nearest
peak so we start on-mode but at the wrong mode.

In [ ]:
# Create a shifted starting point
rng_shift = np.random.default_rng(123)

x_shifted = p5_truth.copy()
mode_offsets = []

print("Shifting distances to wrong modes (within 2-sigma of prior)...")
for j in range(Npulsars):
    mu_j = float(dist_mu[j])
    sig_j = float(dist_sig[j])
    dL_j = dL_truth[j]
    
    max_modes_up = int((mu_j + 2*sig_j - p5_truth[8+j]) / dL_j)
    max_modes_dn = int((p5_truth[8+j] - max(mu_j - 2*sig_j, dL_j)) / dL_j)
    lo = min(-max_modes_dn, -3)
    hi = max(max_modes_up, 3)
    n_shift = rng_shift.integers(lo, hi+1)
    if n_shift == 0:
        n_shift = rng_shift.choice([-3, 3])
    
    x_shifted[8+j] += n_shift * dL_j
    x_shifted[8+j] = np.clip(x_shifted[8+j], dL_j, mu_j + 2*sig_j)
    mode_offsets.append(n_shift)

# Snap each distance to the nearest actual mode peak
print("Snapping to nearest mode peaks...")
for j in range(Npulsars):
    d_new = snap_to_peak(x_shifted, j, float(dL_truth[j]), n_pts=200)
    x_shifted[8+j] = d_new

lp_shifted = float(logp(jnp.array(x_shifted)))
print(f"\nShifted starting point:")
print(f"  logp = {lp_shifted:.2f}  (truth = {lp_truth:.2f}, delta = {lp_shifted - lp_truth:.2f})")
for j in range(Npulsars):
    err_dL = abs(x_shifted[8+j] - p5_truth[8+j]) / dL_truth[j]
    nsig = (x_shifted[8+j] - float(dist_mu[j])) / float(dist_sig[j])
    print(f"  {disco_psrs[j].name}: err = {err_dL:.0f} dL from truth, "
          f"{nsig:+.2f} sigma from prior")

# Compute Fisher at the shifted starting point (not truth!)
print(f"\nComputing Fisher at shifted start:")
fisher_s = compute_fisher(x_shifted, logp)

# Run sampler
print(f"\nRunning sampler from shifted start...")
dL_shifted = np.array(compute_delta_L(x_shifted[0], x_shifted[1], x_shifted[4]))

chain2, lps2, ar2 = run_sampler(
    logp, x_shifted.copy(), np.random.default_rng(42),
    fisher_dict=fisher_s, dL_arr=dL_shifted, mu_arr=mu_arr, sig_arr=sd_arr,
    n_burn=5000, n_prod=20000, report_every=5000,
    hmc_L=20, hmc_eps_init=0.1, hmc_target_ar=0.65)

cov2, ess2 = run_diagnostics(chain2, lps2, ar2, p5_truth, dL_truth,
                              title='Test 2: Shifted start (HMC L=20)',
                              show_start=x_shifted)

In [ ]:
# Side-by-side comparison
print(f"{'='*70}")
print(f"{'':25s} {'Test 1 (truth)':>18s} {'Test 2 (shifted)':>18s}")
print(f"{'='*70}")
print(f"{'Coverage':25s} {cov1:>13d}/13   {cov2:>13d}/13")
print(f"{'ESS min':25s} {min(ess1.values()):>16.0f}   {min(ess2.values()):>16.0f}")
print(f"{'ESS max':25s} {max(ess1.values()):>16.0f}   {max(ess2.values()):>16.0f}")
print(f"{'HMC AR':25s} {ar1['hmc']:>16.4f}   {ar2['hmc']:>16.4f}")
print(f"{'Freq-dist AR':25s} {ar1['freq_dist']:>16.4f}   {ar2['freq_dist']:>16.4f}")
print(f"{'Dist within AR':25s} {ar1['dist_within']:>16.4f}   {ar2['dist_within']:>16.4f}")
print(f"{'Big jump AR':25s} {ar1['big_jump']:>16.4f}   {ar2['big_jump']:>16.4f}")
print(f"{'='*70}")

print(f"\nPer-pulsar distance errors (dL from truth):")
print(f"  {'Pulsar':13s} {'Test 1':>10s} {'Test 2':>10s}")
for j in range(Npulsars):
    med1 = np.median(chain1[:, 8+j])
    med2 = np.median(chain2[:, 8+j])
    err1 = abs(med1 - p5_truth[8+j]) / dL_truth[j]
    err2 = abs(med2 - p5_truth[8+j]) / dL_truth[j]
    print(f"  {disco_psrs[j].name:13s} {err1:>10.1f} {err2:>10.1f}")